In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


## Baseline results (for reference)

All train & all test
 - Test F1 Score:  0.6338
 - Test Precision: 0.6512
 - Test Recall:    0.6393
 - Test Accuracy:  0.6393

Masking train & all test
 - Test F1 Score:  0.5921
 - Test Precision: 0.6287
 - Test Recall:    0.6066
 - Test Accuracy:  0.6066

All train & Masking test
 - Test F1 Score:  0.6847
 - Test Precision: 0.7844
 - Test Recall:    0.7049
 - Test Accuracy:  0.7049

Masking train & Masking test
 - Test F1 Score:  0.6304
 - Test Precision: 0.6581
 - Test Recall:    0.6393
 - Test Accuracy:  0.6393

## Improvements
All train & all test
 - Test F1 Score:  0.6539
 - Test Precision: 0.6610
 - Test Recall:    0.6557
 - Test Accuracy:  0.6557

Masking train & all test
 - Test F1 Score:  0.5902
 - Test Precision: 0.5905
 - Test Recall:    0.5902
 - Test Accuracy:  0.5902

All train & Masking test
 - Test F1 Score:  0.6711
 - Test Precision: 0.6758
 - Test Recall:    0.6721
 - Test Accuracy:  0.6721

Masking train & Masking test
 - Test F1 Score:  0.6850
 - Test Precision: 0.7002
 - Test Recall:    0.6885
 - Test Accuracy:  0.6885

# Training Walkthrough

Same staged pipeline as `train_3d_vit()` in `train_model.py`.

1. Configure the experiment (all params match `config.py` defaults).
2. SSL pretraining on BraTS tensors.
3. Multimodal survival training and testing.

In [2]:
from maikol_utils.print_utils import print_separator
from src.config import Configuration
from src.training import train_stage_ssl, train_stage_survival, test_model
from src.utils import set_all_seeds
from scripts import prepare_data

CONFIG = Configuration(
    ssl_epochs=30,
    survival_epochs=20,
    masked_train=True,
    masked_test=True,
    pos_embed="1d",
    use_radiomics=False,
    dynamic_dropout=False,
    ssl_dataset="brats",
)

set_all_seeds(CONFIG.seed)


# Prepare data

In [3]:
print_separator("Preparing Data", sep_type='LONG')
ssl_train_loader, survival_dm, train_loader, val_loader, test_loader = prepare_data(CONFIG)

## Step 1: SSL Pretraining

Contrastive SSL on BraTS tensors. Trains from scratch (no ViT pretraining step).

In [4]:
print_separator("Starting SSL Pretraining", sep_type='SUPER')
ssl_checkpoint_path = train_stage_ssl(
    CONFIG,
    ssl_train_loader,
)

## Step 2: Survival Training

Loads the best SSL checkpoint (selected by val_loss) into the multimodal survival model.

In [5]:
print_separator("Starting Survival Training", sep_type='SUPER')
survival_module = train_stage_survival(
    CONFIG, ssl_checkpoint_path,
    survival_dm, train_loader,
    val_loader, test_loader
)

## Step 3: Testing

Separate test step on the test loader.

In [6]:
print_separator("Step 3: Testing", sep_type='SUPER')
results = test_model(CONFIG, survival_module, test_loader)

In [ ]:
results['predictions']